# 2. Exploratory Data Analysis & Feature Engineering
**AAI-540 MLOps Project — Group 6: Student Performance Prediction**

This notebook covers:
- Distribution analysis of all features
- Correlation analysis and heatmaps
- Feature importance exploration
- Data preprocessing (encoding, scaling)
- Feature engineering and selection
- SageMaker Feature Store integration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
df = pd.read_csv('../data/processed/student_combined.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# G3 distribution
axes[0].hist(df['G3'], bins=20, edgecolor='black', color='steelblue', alpha=0.7)
axes[0].axvline(x=12, color='red', linestyle='--', linewidth=2, label='Threshold (12)')
axes[0].set_xlabel('Final Grade (G3)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Final Grades (G3)')
axes[0].legend()

# Performance class distribution
perf_counts = df['performance'].value_counts()
axes[1].bar(['Low (0)', 'High (1)'], [perf_counts[0], perf_counts[1]], 
            color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[1].set_xlabel('Performance Category')
axes[1].set_ylabel('Count')
axes[1].set_title('Target Variable Distribution')
for i, v in enumerate([perf_counts[0], perf_counts[1]]):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.2 Numerical Feature Distributions

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols_plot = [c for c in numerical_cols if c not in ['performance']]
print(f'Numerical columns: {numerical_cols_plot}')

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(numerical_cols_plot):
    if i < len(axes):
        df[col].hist(ax=axes[i], bins=20, edgecolor='black', color='steelblue', alpha=0.7)
        axes[i].set_title(col, fontsize=11)
        axes[i].tick_params(labelsize=9)

for j in range(len(numerical_cols_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Numerical Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 Categorical Feature Distributions

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns: {categorical_cols}')

fig, axes = plt.subplots(3, 5, figsize=(22, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if i < len(axes):
        df[col].value_counts().plot(kind='bar', ax=axes[i], color='steelblue', 
                                     edgecolor='black', alpha=0.7)
        axes[i].set_title(col, fontsize=11)
        axes[i].tick_params(labelsize=9, rotation=45)

for j in range(len(categorical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Categorical Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Correlation Analysis

In [ ]:
plt.figure(figsize=(16, 12))
corr_matrix = df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap - Numerical Features', fontsize=16)
plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=== Top Correlations with Final Grade (G3) ===')
g3_corr = corr_matrix['G3'].drop('G3').sort_values(ascending=False)
print(g3_corr)

## 2.5 Feature vs Target Analysis

In [ ]:
key_features = ['studytime', 'failures', 'absences', 'Medu', 'Fedu', 'age', 'goout', 'Dalc', 'Walc', 'health']

fig, axes = plt.subplots(2, 5, figsize=(24, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    sns.boxplot(x='performance', y=col, data=df, ax=axes[i], palette=['#e74c3c', '#2ecc71'])
    axes[i].set_title(f'{col} vs Performance', fontsize=11)
    axes[i].set_xticklabels(['Low', 'High'])

plt.suptitle('Key Features vs Performance Category', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/features_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cat_features_analysis = ['school', 'sex', 'address', 'Pstatus', 'higher', 'internet', 'romantic', 'subject']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features_analysis):
    ct = pd.crosstab(df[col], df['performance'], normalize='index')
    ct.plot(kind='bar', ax=axes[i], color=['#e74c3c', '#2ecc71'], 
            edgecolor='black', alpha=0.8)
    axes[i].set_title(f'{col} vs Performance', fontsize=11)
    axes[i].set_ylabel('Proportion')
    axes[i].legend(['Low', 'High'], fontsize=9)
    axes[i].tick_params(rotation=0)

plt.suptitle('Categorical Features vs Performance Category', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/categorical_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.6 Grade Progression Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# G1 vs G3
axes[0].scatter(df['G1'], df['G3'], alpha=0.3, c=df['performance'], 
                cmap='RdYlGn', edgecolors='gray', s=30)
axes[0].set_xlabel('First Period Grade (G1)')
axes[0].set_ylabel('Final Grade (G3)')
axes[0].set_title('G1 vs G3 (colored by performance)')

# G2 vs G3
axes[1].scatter(df['G2'], df['G3'], alpha=0.3, c=df['performance'], 
                cmap='RdYlGn', edgecolors='gray', s=30)
axes[1].set_xlabel('Second Period Grade (G2)')
axes[1].set_ylabel('Final Grade (G3)')
axes[1].set_title('G2 vs G3 (colored by performance)')

plt.tight_layout()
plt.savefig('../data/processed/grade_progression.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Feature Engineering & Preprocessing

In [ ]:
# Drop G1, G2, G3 — G1/G2 are intermediate grades that leak the target, G3 is used to create target
features_to_drop = ['G1', 'G2', 'G3']
df_features = df.drop(columns=features_to_drop)

print(f'Shape after dropping grade columns: {df_features.shape}')
print(f'Remaining columns: {list(df_features.columns)}')

In [ ]:
# Encode binary categorical variables
binary_map = {
    'school': {'GP': 0, 'MS': 1},
    'sex': {'F': 0, 'M': 1},
    'address': {'U': 0, 'R': 1},
    'famsize': {'LE3': 0, 'GT3': 1},
    'Pstatus': {'T': 0, 'A': 1},
    'schoolsup': {'no': 0, 'yes': 1},
    'famsup': {'no': 0, 'yes': 1},
    'paid': {'no': 0, 'yes': 1},
    'activities': {'no': 0, 'yes': 1},
    'nursery': {'no': 0, 'yes': 1},
    'higher': {'no': 0, 'yes': 1},
    'internet': {'no': 0, 'yes': 1},
    'romantic': {'no': 0, 'yes': 1}
}

for col, mapping in binary_map.items():
    df_features[col] = df_features[col].map(mapping)

print('Binary encoding complete.')

In [ ]:
# One-hot encode multi-class categorical variables
multi_cat_cols = ['Mjob', 'Fjob', 'reason', 'guardian', 'subject']
df_encoded = pd.get_dummies(df_features, columns=multi_cat_cols, drop_first=True, dtype=int)

print(f'Shape after one-hot encoding: {df_encoded.shape}')
print(f'\nNew columns added:')
new_cols = [c for c in df_encoded.columns if c not in df_features.columns]
print(new_cols)

In [ ]:
# Feature Scaling — scale numerical features
num_cols_to_scale = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
                     'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']

scaler = StandardScaler()
df_encoded[num_cols_to_scale] = scaler.fit_transform(df_encoded[num_cols_to_scale])

print('Feature scaling complete.')
df_encoded[num_cols_to_scale].describe().round(3)

In [ ]:
# Final check
print(f'Final dataset shape: {df_encoded.shape}')
print(f'\nTarget distribution:')
print(df_encoded['performance'].value_counts())
print(f'\nAny nulls remaining: {df_encoded.isnull().sum().sum()}')
df_encoded.head()

## 2.8 Save Processed Data

In [ ]:
# Save the fully processed dataset
df_encoded.to_csv('../data/processed/student_processed.csv', index=False)
print('Processed dataset saved to: ../data/processed/student_processed.csv')

# Also save the scaler for later use
import joblib
joblib.dump(scaler, '../data/processed/scaler.joblib')
print('Scaler saved to: ../data/processed/scaler.joblib')

## 2.9 SageMaker Feature Store (AWS)
Uncomment when AWS access is available.

In [ ]:
# import sagemaker
# from sagemaker.session import Session
# from sagemaker.feature_store.feature_group import FeatureGroup
# import time
#
# session = sagemaker.Session()
# region = session.boto_region_name
# bucket = session.default_bucket()
# role = sagemaker.get_execution_role()
#
# # Prepare data for feature store
# df_feature_store = df_encoded.copy()
# df_feature_store['record_id'] = df_feature_store.index.astype(str)
# df_feature_store['event_time'] = pd.Timestamp.now().isoformat()
#
# # Create Feature Group
# feature_group_name = f'student-performance-features-{int(time.time())}'
# feature_group = FeatureGroup(name=feature_group_name, sagemaker_session=session)
#
# feature_group.load_feature_definitions(data_frame=df_feature_store)
#
# feature_group.create(
#     s3_uri=f's3://{bucket}/feature-store/',
#     record_identifier_name='record_id',
#     event_time_feature_name='event_time',
#     role_arn=role,
#     enable_online_store=True
# )
#
# # Wait for feature group to be created
# feature_group.describe()
#
# # Ingest data
# feature_group.ingest(data_frame=df_feature_store, max_workers=3, wait=True)
# print(f'Feature group created: {feature_group_name}')
# print(f'Records ingested: {len(df_feature_store)}')

## Summary
### EDA Findings:
- G1 and G2 are highly correlated with G3 (>0.8) — dropped to avoid data leakage
- `failures` has strong negative correlation with performance
- `studytime` and parental education (`Medu`, `Fedu`) positively correlate with grades
- Students wanting higher education show significantly better performance
- Alcohol consumption (`Dalc`, `Walc`) negatively impacts grades

### Feature Engineering:
- Binary encoded 13 categorical features
- One-hot encoded 5 multi-class features
- Scaled 13 numerical features using StandardScaler
- Final processed dataset ready for model training